# PostgreSQL Data Validation

## Project: Germany's Gender Pay Gap Across Industries (2007–2024)

### Objective
I use this notebook to validate the staging view created by dbt in PostgreSQL.

The validation checks include:
- Total number of observations
- Year coverage
- Minimum and maximum gender pay gap
- Comparison with the original raw dataset

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
# Establish a connection to my PostgreSQL database

username = "chibuikemaduka"
host = "localhost"
port = "5432"
database = "gender_pay_gap"

connection_url = f"postgresql+psycopg2://{username}@{host}:{port}/{database}"

engine = create_engine(connection_url)

In [4]:
# Test the PostgreSQL connection

with engine.connect() as connection:
    result = connection.execute(text("SELECT version();"))
    print(result.scalar())

PostgreSQL 18.6 (Homebrew) on aarch64-apple-darwin25.6.0, compiled by Apple clang version 21.0.0 (clang-2100.3.34.2), 64-bit


In [5]:
# Count the observations in my dbt staging view

query = """
SELECT COUNT(*) AS total_records
FROM public.stg_gender_pay_gap;
"""

with engine.connect() as connection:
    record_count = pd.read_sql(text(query), connection)

record_count

,total_records
0,372


In [6]:
# Check the year coverage and gender pay gap range

query = """
SELECT
    MIN(year) AS first_year,
    MAX(year) AS last_year,
    MIN(gender_pay_gap_percent) AS minimum_gap,
    MAX(gender_pay_gap_percent) AS maximum_gap
FROM public.stg_gender_pay_gap;
"""

with engine.connect() as connection:
    summary = pd.read_sql(text(query), connection)

summary

,first_year,last_year,minimum_gap,maximum_gap
0,2007,2024,-2.2,34.2


In [7]:

# Compare the number of records in my raw table and staging view

query = """
SELECT
    (SELECT COUNT(*) FROM public.raw_gender_pay_gap) AS raw_records,
    (SELECT COUNT(*) FROM public.stg_gender_pay_gap) AS staging_records;
"""

with engine.connect() as connection:
    comparison = pd.read_sql(text(query), connection)

comparison["counts_match"] = (
    comparison["raw_records"] == comparison["staging_records"]
)

comparison


,raw_records,staging_records,counts_match
0,372,372,True


## Duplicate Records Check

In [8]:

# Check for duplicate country, year and industry combinations in my staging view

query = """
-- Identify combinations that appear more than once
SELECT
    country,
    year,
    industry,
    COUNT(*) AS duplicate_count
FROM public.stg_gender_pay_gap
GROUP BY
    country,
    year,
    industry
HAVING COUNT(*) > 1
ORDER BY year, industry;
"""

with engine.connect() as connection:
    duplicates = pd.read_sql(text(query), connection)

print("Number of duplicate combinations:", len(duplicates))

duplicates


Number of duplicate combinations: 0


,country,year,industry,duplicate_count


## Missing Values Check

In [9]:

# Check for missing values in all columns of my staging view

query = """
-- Count missing values in each column
SELECT
    COUNT(*) FILTER (WHERE frequency IS NULL) AS missing_frequency,
    COUNT(*) FILTER (WHERE unit IS NULL) AS missing_unit,
    COUNT(*) FILTER (WHERE industry IS NULL) AS missing_industry,
    COUNT(*) FILTER (WHERE country IS NULL) AS missing_country,
    COUNT(*) FILTER (WHERE year IS NULL) AS missing_year,
    COUNT(*) FILTER (WHERE gender_pay_gap_percent IS NULL) AS missing_pay_gap
FROM public.stg_gender_pay_gap;
"""

with engine.connect() as connection:
    missing_values = pd.read_sql(text(query), connection)

missing_values


,missing_frequency,missing_unit,missing_industry,missing_country,missing_year,missing_pay_gap
0,0,0,0,0,0,0


## Final Validation Summary

I successfully validated the dbt staging view against my original PostgreSQL raw dataset.

The validation covered record counts, year coverage, gender pay gap ranges, duplicate combinations and missing values.

All checks passed, confirming that the staging view preserves the structure and records of my original Eurostat dataset.

In [10]:

# Summarise the results of my PostgreSQL staging validation

validation_summary = pd.DataFrame({
    "Validation": [
        "Raw record count",
        "Staging record count",
        "Record counts match",
        "Year coverage",
        "Gender pay gap range",
        "Duplicate combinations",
        "Missing values"
    ],
    "Result": [
        372,
        372,
        "True",
        "2007–2024",
        "-2.20% to 34.20%",
        0,
        0
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

validation_summary


,Validation,Result,Status
0,Raw record count,372,PASS
1,Staging record count,372,PASS
2,Record counts match,True,PASS
3,Year coverage,2007–2024,PASS
4,Gender pay gap range,-2.20% to 34.20%,PASS
5,Duplicate combinations,0,PASS
6,Missing values,0,PASS


## Inspecting Industry Categories

In [11]:

# Retrieve all unique industry categories from my dbt staging view

query = """
-- List the distinct economic activity categories in the dataset
SELECT DISTINCT
    industry
FROM public.stg_gender_pay_gap
ORDER BY industry;
"""

with engine.connect() as connection:
    industries = pd.read_sql(text(query), connection)

print("Number of unique industry categories:", len(industries))

industries


Number of unique industry categories: 21


,industry
0,Accommodation and food service activities
1,Administrative and support service activities
2,"Arts, entertainment and recreation"
3,Business economy
4,Construction
5,Education
6,"Electricity, gas, steam and air conditioning s..."
7,Financial and insurance activities
8,Human health and social work activities
9,"Industry, construction and services (except ac..."


In [12]:
# Display the complete industry names without truncation

print(industries.to_string(index=False))

                                                                                                                         industry
                                                                                        Accommodation and food service activities
                                                                                    Administrative and support service activities
                                                                                               Arts, entertainment and recreation
                                                                                                                 Business economy
                                                                                                                     Construction
                                                                                                                        Education
                                                                              Electricity,

## Validating the Intermediate Model

In [13]:
# Check the record count and industry coverage in my intermediate model

query = """
-- Summarise the records and distinct industries after excluding aggregates
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT industry) AS unique_industries,
    MIN(year) AS first_year,
    MAX(year) AS last_year,
    ROUND(AVG(gender_pay_gap_percent), 2) AS average_gap
FROM public.int_industry_pay_gap;
"""

with engine.connect() as connection:
    intermediate_summary = pd.read_sql(text(query), connection)

intermediate_summary

,total_records,unique_industries,first_year,last_year,average_gap
0,321,18,2007,2024,17.05


## Validating Annual Pay Gap Trends

In [14]:
# Retrieve annual gender pay gap statistics from my dbt mart

query = """
-- Review annual averages, medians and industry coverage
SELECT
    year,
    industry_count,
    observation_count,
    average_pay_gap,
    median_pay_gap,
    lowest_pay_gap,
    highest_pay_gap
FROM public.mart_pay_gap_by_year
ORDER BY year;
"""

with engine.connect() as connection:
    annual_trends = pd.read_sql(text(query), connection)

annual_trends

,year,industry_count,observation_count,average_pay_gap,median_pay_gap,lowest_pay_gap,highest_pay_gap
0,2007,17,17,20.15,21.50,5.1,34.2
1,2008,17,17,20.12,21.40,6.4,33.9
2,2009,17,17,19.95,21.50,4.9,34.0
3,2010,18,18,18.86,20.95,3.9,33.7
4,2011,18,18,18.77,19.60,5.0,32.8
5,2012,18,18,18.48,19.25,3.5,33.6
6,2013,18,18,17.96,19.35,4.1,31.4
7,2014,18,18,17.86,18.75,0.4,31.3
8,2015,18,18,17.29,17.45,1.4,31.3
9,2016,18,18,16.81,17.80,2.4,30.6


## Validating Industry-Level Pay Gap Statistics

In [15]:
# Retrieve gender pay gap statistics for each industry

query = """
-- Rank individual industries by their average gender pay gap
SELECT
    industry,
    observation_count,
    first_year,
    last_year,
    average_pay_gap,
    median_pay_gap,
    lowest_pay_gap,
    highest_pay_gap
FROM public.mart_pay_gap_by_industry
ORDER BY average_pay_gap DESC;
"""

with engine.connect() as connection:
    industry_analysis = pd.read_sql(text(query), connection)

industry_analysis

,industry,observation_count,first_year,last_year,average_pay_gap,median_pay_gap,lowest_pay_gap,highest_pay_gap
0,"Professional, scientific and technical activities",18,2007,2024,30.29,30.50,25.2,34.2
1,"Arts, entertainment and recreation",18,2007,2024,27.50,28.80,19.6,31.3
2,Financial and insurance activities,18,2007,2024,27.18,27.35,23.2,30.1
3,Information and communication,18,2007,2024,25.49,25.25,19.3,28.9
4,Manufacturing,18,2007,2024,24.56,25.00,19.1,28.9
5,Human health and social work activities,18,2007,2024,23.24,23.85,18.7,24.6
6,Wholesale and retail trade; repair of motor ve...,18,2007,2024,22.92,22.55,19.3,25.8
7,"Electricity, gas, steam and air conditioning s...",18,2007,2024,20.06,21.40,13.8,22.9
8,Other service activities,18,2007,2024,19.78,19.65,12.2,26.4
9,Real estate activities,18,2007,2024,16.19,16.35,8.4,21.5


## Comparing Industry Pay Gaps: 2007 vs 2024

In [16]:
# Compare gender pay gaps across industries between 2007 and 2024

query = """
-- Retrieve the baseline and latest pay gap for each industry
SELECT
    industry,
    gap_2007,
    gap_2024,
    change_percentage_points
FROM public.mart_pay_gap_comparison
ORDER BY change_percentage_points ASC NULLS LAST;
"""

with engine.connect() as connection:
    comparison = pd.read_sql(text(query), connection)

comparison

,industry,gap_2007,gap_2024,change_percentage_points
0,Other service activities,26.3,13.8,-12.5
1,"Water supply; sewerage, waste management and r...",8.3,-2.2,-10.5
2,Manufacturing,28.9,19.1,-9.8
3,Information and communication,28.9,19.3,-9.6
4,"Professional, scientific and technical activities",34.2,25.2,-9.0
5,"Electricity, gas, steam and air conditioning s...",21.5,13.8,-7.7
6,Accommodation and food service activities,13.7,6.2,-7.5
7,Real estate activities,19.6,12.2,-7.4
8,"Arts, entertainment and recreation",27.7,20.6,-7.1
9,Wholesale and retail trade; repair of motor ve...,25.3,19.3,-6.0
